In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc PyGithub networkx qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Iskay Quantum Optimizer – funkce Qiskit od Kipu Quantum

> **Note:** * Funkce Qiskit jsou experimentální funkce dostupné pouze uživatelům plánů IBM Quantum&reg; Premium, Flex a On-Prem (prostřednictvím rozhraní IBM Quantum Platform API). Jsou ve stavu preview release a mohou se změnit.

## Přehled
S Iskay Quantum Optimizerem od Kipu Quantum můžeš řešit složité optimalizační úlohy na kvantových počítačích IBM&reg;. Tento solver využívá špičkový [algoritmus bf-DCQO](https://doi.org/10.48550/arXiv.2409.04477) od Kipu, který jako vstup vyžaduje pouze účelovou funkci a automaticky poskytuje řešení daného problému. Dokáže zpracovat optimalizační úlohy zahrnující až 156 qubitů, čímž umožňuje využít všechny qubity kvantových zařízení IBM. Optimizer používá mapování 1:1 mezi klasickými proměnnými a qubity, což ti umožňuje řešit optimalizační úlohy s až 156 binárními proměnnými.

Optimizer umožňuje řešení neomezených binárních optimalizačních problémů. Kromě běžně používané formulace QUBO (Quadratic Unconstrained Binary Optimization) podporuje také optimalizační problémy vyššího řádu (HUBO). Solver využívá nevariační kvantový algoritmus a většinu výpočtů provádí na kvantových zařízeních.

Níže najdeš podrobnější informace o použitém algoritmu a stručný návod k použití funkce, spolu s výsledky benchmarkování na různých instancích problémů různých velikostí a složitostí.

## Popis
Optimizer je připravená implementace špičkových kvantových optimalizačních algoritmů. Řeší optimalizační problémy spouštěním vysoce komprimovaných kvantových Circuit na kvantovém hardware. Tato komprese je dosažena zavedením kontradiabatických členů do základní časové evoluce kvantového systému. Algoritmus provádí několik iterací běhů na hardware, aby získal konečná řešení, a kombinuje je s následným zpracováním. Tyto kroky jsou plynule integrovány do pracovního postupu Optimizeru a jsou prováděny automaticky.

### Jak Quantum Optimizer funguje?
Tato část popisuje základy implementovaného algoritmu bf-DCQO. Úvod do algoritmu najdeš také na [kanálu Qiskit na YouTube.](https://www.youtube.com/watch?v=33QmsXhIlpU&t=1223s)

Algoritmus je založen na časové evoluci kvantového systému, který se v čase transformuje, přičemž řešení problému je zakódováno v základním stavu kvantového systému na konci evoluce. Podle [adiabatického teorému](https://en.wikipedia.org/wiki/Adiabatic_theorem) musí být tato evoluce pomalá, aby se zajistilo, že systém zůstane ve svém základním stavu. Digitalizace této evoluce je základem digitalizovaného kvantového adiabatického výpočtu (DQA) a nechvalně známého algoritmu QAOA. Pomalá evoluce však není pro rostoucí velikosti problémů proveditelná, protože vede k rostoucí hloubce Circuit. Pomocí kontradiabatických protokolů lze potlačit nežádoucí excitace vznikající při krátkých dobách evoluce a přitom zůstat v základním stavu. Digitalizace této kratší doby evoluce zde vede ke kvantovým Circuit s menší hloubkou a méně zapletovacími Gate.

Circuit algoritmů bf-DCQO obvykle využívají až desetkrát méně zapletovacích Gate než DQA a třikrát až čtyřikrát méně zapletovacích Gate než standardní implementace QAOA. Díky menšímu počtu Gate dochází při provádění Circuit na hardware k méně chybám. Optimizer proto nevyžaduje použití technik jako potlačení nebo zmírnění chyb. Jejich implementace v budoucích verzích může dále zlepšit kvalitu řešení.

Ačkoli algoritmus bf-DCQO používá iterace, je nevariační. Po každé iteraci algoritmu je měřeno rozdělení stavů. Získané rozdělení se používá k výpočtu tzv. pole předpojatosti (bias-field). Toto pole umožňuje zahájit další iteraci z energetického stavu blízkého dříve nalezenému řešení. Tímto způsobem algoritmus s každou iterací postupuje k řešením nižší energie. Typicky postačuje přibližně deset iterací ke konvergenci k řešení, celkem tedy vyžaduje podstatně nižší počet iterací než variační algoritmy, jejichž počet se pohybuje v řádu přibližně 100 iterací.

Optimizer kombinuje algoritmus bf-DCQO s klasickým následným zpracováním. Po naměření rozdělení stavů se provede lokální prohledávání. Během lokálního prohledávání jsou bity naměřeného řešení náhodně přehazovány. Po přehození je vyhodnocena energie nového bitového řetězce. Pokud je energie nižší, je bitový řetězec ponechán jako nové řešení. Lokální prohledávání roste pouze lineárně s počtem qubitů, takže je výpočetně nenáročné. Protože následné zpracování opravuje lokální bitové přehody, kompenzuje chyby bitových přechodů, které jsou často důsledkem nedokonalosti hardware a chyb při čtení.

### Pracovní postup
Níže je schéma pracovního postupu Quantum Optimizeru.

![Pracovní postup](../docs/images/guides/kipu-optimization/workflow.svg "Pracovní postup Quantum Optimizeru")

Použitím Quantum Optimizeru lze řešení optimalizačního problému na kvantovém hardware zredukovat na:
* Formulaci účelové funkce problému
* Přístup k Optimizeru přes Qiskit Functions
* Spuštění Optimizeru a sběr výsledků

## Benchmarky
Níže uvedené metriky benchmarků ukazují, že Optimizer efektivně řeší problémy zahrnující až 156 qubitů a nabízí obecný přehled přesnosti a škálovatelnosti optimizeru napříč různými typy problémů. Skutečné výkonnostní metriky se mohou lišit v závislosti na konkrétních vlastnostech problému, jako je počet proměnných, hustota a lokalita členů v účelové funkci a polynomiální řád.

Následující tabulka obsahuje aproximační poměr (AR), metriku definovanou takto:
$$
AR = \frac{C^{*} - C_\textrm{max}}{C_{\textrm{min}} - C_{\textrm{max}}},
$$
kde $C$ je účelová funkce, $C_{\textrm{min}}$, $C_{\textrm{max}}$ jsou její minimální a maximální hodnoty a $C^{*}$ je cena nejlepšího nalezeného řešení. AR=100\% tedy znamená, že byl nalezen základní stav problému.

| Příklad           | Počet qubitů | Aproximační poměr | Celkový čas (s) | Využití runtime (s) | Celkový počet shotů | Počet iterací |
| ----------------- | :----------: | :---------------: | :-------------: | :-----------------: | :-----------------: | :-----------: |
| Unweighted MaxCut |      28      |       100%        |       180       |         30          |         30k         |       5       |
| Unweighted MaxCut |      30      |       100%        |       180       |         30          |         30k         |       5       |
| Unweighted MaxCut |      32      |       100%        |       180       |         30          |         30k         |       5       |
| Unweighted MaxCut |      80      |       100%        |       480       |         60          |         90k         |       9       |
| Unweighted MaxCut |     100      |       100%        |       330       |         60          |         60k         |       6       |
| Unweighted MaxCut |     120      |       100%        |       370       |         60          |         60k         |       6       |
| HUBO 1            |     156      |       100%        |       600       |         70          |        100k         |      10       |
| HUBO 2            |     156      |       100%        |       600       |         70          |        100k         |      10       |

- Instance MaxCut s 28, 30 a 32 qubity byly spuštěny na ibm_sherbrooke. Instance s 80, 100 a 120 byly spuštěny na procesoru Heron r2.
- Instance HUBO byly také spuštěny na procesoru Heron r2.

Všechny instance benchmarků jsou dostupné na GitHubu (viz [instance benchmarků Kipu](https://github.com/Kipu-Quantum-GmbH/benchmark-instances)). Příklad ke spuštění těchto instancí najdeš v části [Příklad 3: Instance benchmarků](#example-3-benchmark-instances).

## Vstupy a výstupy
### Vstup
V následující tabulce najdeš všechny vstupní parametry, které Quantum Optimizer přijímá. Následující část **Možnosti** podrobněji popisuje dostupné `options`.

| Název        | Typ                | Popis                                                                                                                                                                                                           | Povinný | Výchozí hodnota                                                                                             | Příklad                                                                                                              |
| ------------ | ------------------ | :-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ------- | ----------------------------------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------------- |
| problem      | `Dict[str, float]` | Koeficienty optimalizačního problému formulovaného ve formátu QUBO/HUBO nebo spin. Více informací o specifikaci problému najdeš v části [Přijímané formáty problémů](#accepted-problem-formats)                  | Ano     | N/A                                                                                                         | `{"()": -21.0, "(0, 4)": 0.5,"(0, 2)": 0.5,"(0, 1)": 0.5,"(1, 3)": 0.5}`                                           |
| problem_type | `str`              | Urči, zda jsou koeficienty problému v binárním formátu (QUBO/HUBO) nebo spin formátu. Dvě možnosti jsou `"spin"` nebo `"binary"`                                                                                 | Ano     | N/A                                                                                                         | `"spin"`                                                                                                             |
| backend_name | `str`              | Název Backend pro zadání dotazu                                                                                                                                                                                  | Ano     | N/A                                                                                                         | `"ibm_fez"`                                                                                                          |
| options      | `Dict[str, Any]`   | Možnosti pro správu odesílání na hardware, například počet shotů. Podrobnosti o konfiguraci možností najdeš v části [Možnosti](#options)                                                                         | Ne      | Výchozí hodnoty konfigurace možností najdeš v části [Možnosti](#options)                                    | `{"shots": 5000, "num_iterations": 3, "use_session": True, "seed_transpiler": 42}`                                   |

#### Přijímané formáty problémů
Argumenty `problem` a `problem_type` kódují optimalizační problém ve tvaru

$$
\begin{align}
\min_{(x_1, x_2, \ldots, x_n) \in D} C(x_1, x_2, \ldots, x_n) \nonumber
\end{align}
$$
kde

$$
C(x_1, ... , x_n) = a + \sum_{i} b_i x_i + \sum_{i, j} c_{i, j} x_i x_j + ... + \sum_{k_1, ..., k_m} g_{k_1, ..., k_m} x_{k_1} ... x_{k_m}
$$

- Volbou `problem_type = "binary"` určuješ, že účelová funkce je v `binary` formátu, což znamená $D = {0,  1}^{n}$, tedy účelová funkce je zapsána ve formulaci QUBO/HUBO.
- Naopak volbou `problem_type = "spin"` je účelová funkce zapsána v Isingově formulaci, kde $D = {-1, 1}^{n}$.

Koeficienty problému by měly být zakódovány ve slovníku takto:
$$
\begin{align} \nonumber
&\texttt{{} \\ \nonumber
&\texttt{"()"}&: \quad &a, \\ \nonumber
&\texttt{"(i,)"}&: \quad &b_i, \\ \nonumber
&\texttt{"(i, j)"}&: \quad &c_{i, j}, \\ \nonumber
&\quad  \vdots \\ \nonumber
&\texttt{"(} k_1, ..., k_m  \texttt{)"} &: \quad &g_{k_1, ..., k_m}, \\ \nonumber
&\texttt{}}
\end{align}
$$

- Upozorňujeme, že klíče slovníku musí být řetězce obsahující platnou n-tici neopakujících se celých čísel.
#### Možnosti
Iskay poskytuje možnosti jemného doladění prostřednictvím volitelných parametrů. Výchozí hodnoty fungují dobře pro většinu problémů, ale chování lze přizpůsobit konkrétním požadavkům:
| Parametr | Typ | Výchozí hodnota | Popis |
|-----------|------|---------|-------------|
| **shots** | `int` | 10000 | Kvantová měření za iteraci (vyšší = přesnější) |
| **num_iterations** | `int` | 10 | Iterace algoritmu (více iterací může zlepšit kvalitu řešení) |
| **use_session** | `bool` | True | Použití IBM Session pro zkrácení čekacích dob ve frontě |
| **seed_transpiler** | `int` | None | Nastavit pro reprodukovatelnou kompilaci kvantového Circuit |
| **direct_qubit_mapping** | `bool` | False | Mapovat virtuální Qubit přímo na fyzické Qubit |
| **job_tags** | `List[str]` | None | Vlastní štítky pro sledování úloh |
| **preprocessing_level** | `int` | 0 | Intenzita předzpracování problému (0–3) – viz podrobnosti níže |
| **postprocessing_level** | `int` | 2 | Úroveň zdokonalení řešení (0–2) – viz podrobnosti níže |
| **transpilation_level** | `int` | 0 | Počet pokusů o optimalizaci Transpileru (0–5) – viz podrobnosti níže |
| **transpile_only** | `bool` | False | Analyzovat optimalizaci Circuit bez spuštění plného výpočtu |

**Úrovně předzpracování (0–3)**: Zvláště důležité pro větší problémy, které aktuálně nelze vejít do koherenčních časů hardwaru. Vyšší úrovně předzpracování dosahují mělčích hloubek Circuit prostřednictvím aproximací při transpilaci problému:
- **Úroveň 0**: Přesné, delší Circuit
- **Úroveň 1**: Dobrá rovnováha mezi přesností a aproximací – odstraňuje pouze Gate s úhly v nejnižším 10. percentilu
- **Úroveň 2**: Mírně vyšší aproximace – odstraňuje Gate s úhly v nejnižším 20. percentilu a používá `approximation_degree=0.95` při transpilaci
- **Úroveň 3**: Maximální úroveň aproximace – odstraňuje Gate v nejnižším 30. percentilu a používá `approximation_degree=0.90` při transpilaci

**Úrovně transpilace (0–5)**: Řídí pokročilé pokusy o optimalizaci Transpileru při kompilaci kvantového Circuit. To může vést ke zvýšení klasické režie a v některých případech nemusí změnit hloubku Circuit. Výchozí hodnota `2` obecně vede k nejmenšímu Circuit a je relativně rychlá.
- **Úroveň 0**: Optimalizace dekompozovaného DCQO Circuit (rozložení, směrování, plánování)
- **Úroveň 1**: Optimalizace `PauliEvolutionGate` a poté dekompozovaného DCQO Circuit (max_trials=10)
- **Úroveň 2**: Optimalizace `PauliEvolutionGate` a poté dekompozovaného DCQO Circuit (max_trials=15)
- **Úroveň 3**: Optimalizace `PauliEvolutionGate` a poté dekompozovaného DCQO Circuit (max_trials=20)
- **Úroveň 4**: Optimalizace `PauliEvolutionGate` a poté dekompozovaného DCQO Circuit (max_trials=25)
- **Úroveň 5**: Optimalizace `PauliEvolutionGate` a poté dekompozovaného DCQO Circuit (max_trials=50)

**Úrovně následného zpracování (0–2)**: Řídí rozsah klasické optimalizace, která kompenzuje chyby bit-flip různým počtem greedy průchodů lokálního hledání:
- **Úroveň 0**: 1 průchod
- **Úroveň 1**: 2 průchody
- **Úroveň 2**: 3 průchody

**Režim pouze transpilace**: K dispozici pro uživatele, kteří chtějí analyzovat optimalizaci Circuit bez spuštění plného výpočtu kvantového algoritmu.

**Příklad vlastní konfigurace**: Ukázka, jak lze nakonfigurovat Iskay s různými nastaveními:

In [1]:
custom_options = {
    "shots": 15_000,  # Higher shot count for better statistics
    "num_iterations": 12,  # More iterations for solution refinement
    "preprocessing_level": 1,  # Light preprocessing for problem simplification
    "postprocessing_level": 2,  # Maximum postprocessing for solution quality
    "transpilation_level": 3,  # Using higher transpilation level for circuit optimization
    "seed_transpiler": 42,  # Fixed seed for reproducible results
    "job_tags": ["custom_config"],  # Custom tracking tags
}

**Optimalizace seedu**: Vezmi na vědomí, že `seed_transpiler` je ve výchozím nastavení `None`. To umožňuje automatický optimalizační proces Transpileru. Pokud je hodnota `None`, systém spustí pokus s více seedy a vybere ten, který poskytne nejlepší hloubku Circuit – plně využívá parametr `max_trials` pro každou úroveň transpilace.

**Výkon úrovní transpilace**: Zvyšování počtu `max_trials` vyššími hodnotami `transpilation_level` nevyhnutelně prodlužuje dobu transpilace, ale nemusí vždy změnit výsledný Circuit – závisí to silně na konkrétní struktuře a složitosti Circuit. U některých Circuit/problémů však může být rozdíl mezi 10 pokusy (úroveň 1) a 50 pokusy (úroveň 5) dramatický, takže prozkoumání těchto parametrů může být klíčem k úspěšnému nalezení řešení.
### Výstup
| Název   | Typ               | Popis                                                           | Příklad                                                                |
| ------ | ------------------ | ----------------------------------------------------------------------| ---------------------------------------------------------------------- |
| result | `Dict[str, Any]` | Řešení a metadata. Struktura se liší podle možnosti `transpile_only`. |   Viz „Obsah slovníku výsledků" níže      |
### Obsah slovníku výsledků
Struktura slovníku výsledků závisí na režimu spuštění:

| Pole         | Typ               | Režim | Popis                                                                                                             | Příklad |
| ------------ | :------------------: | :--: | ----------------------------------------------------------------------------------------------------------------------- | ------- |
| **solution**     | `Dict[str, int]` | **Standardní** | Seřazené mapované řešení, kde klíče jsou indexy proměnných (jako řetězce) seřazené numericky a hodnoty jsou odpovídající hodnoty proměnných (1/-1 pro spinové problémy, 1/0 pro binární problémy).| `{'0': -1, '1': -1, '2': -1, '3': 1, '4': 1}` |
| **solution_info**      | `Dict[str, Any]`   | **Standardní** | Podrobné informace o řešení (viz podrobnosti níže)                                           | `{'bitstring': '11100', 'cost': -13.8, 'seed_transpiler': 42, 'mapping': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}}` |
| **prob_type** | `str`             | **Standardní** | Typ optimalizačního problému ('spin' nebo 'binary')                                                           | `'spin'` |
| **transpilation_info** | `Dict[str, Any]` | **Pouze transpilace** | Analýza Circuit a podrobnosti o transpilaci (viz podrobnosti níže) | `{'best_seed': 42, 'transpilation_time_seconds': 50.06, 'transpiled_circuit': {'depth': 576, 'gate_count': 4177, 'num_qubits': 156, 'width': 176, 'operations': {'sx': 1325, 'rx': 891, 'cz': 783, 'rz': 650, 'rzz': 466, 'x': 42, 'measure': 20}}}` |

#### Standardní spuštění
Pokud je volitelný parametr `transpile_only=False`:

**Slovník solution_info:**
- **"bitstring"** (`str`): Reprezentace řešení jako surový bitový řetězec.
- **"cost"** (`float`): Hodnota nákladů/energie přidružená k řešení.
- **"seed_transpiler"** (`int`): Náhodný seed použitý pro Transpiler, který vytvořil tento výsledek.
- **"mapping"** (`Dict[int, int]`): Původní mapování Qubit na proměnné použité při výpočtu.
- **"qpu_time"** (`float`, volitelné): Doba provádění na QPU v sekundách.

**Poznámky k mapování proměnných:**
- Slovník `solution` je získán z bitového řetězce řešení za použití objektu `mapping` pro indexování proměnných.
- Pokud `problem_type=spin`, používáme přiřazení $1 \rightarrow -1, \quad 0 \rightarrow 1$.
- Klíče ve slovníku řešení jsou indexy proměnných seřazené numericky jako řetězce.

#### Analýza transpilace
Pokud je volitelný parametr `transpile_only=True`:

**Slovník transpilation_info:**
- **"best_seed"** (`int`): Optimální seed nalezený pro transpilaci
- **"transpilation_time_seconds"** (`float`): Čas potřebný pro proces transpilace
- **"transpiled_circuit"** (`Dict`): Analýza Circuit obsahující:
  - **"depth"** (`int`): Hloubka Circuit (počet vrstev)
  - **"gate_count"** (`int`): Celkový počet Gate v Circuit
  - **"num_qubits"** (`int`): Počet použitých Qubit
  - **"width"** (`int`): Šířka Circuit
  - **"operations"** (`Dict[str, int]`): Počet každého použitého typu Gate

**Použití režimu pouze transpilace:**
- K dispozici pro uživatele, kteří chtějí analyzovat optimalizaci Circuit bez spuštění plného výpočtu kvantového algoritmu.
- Užitečné pro analýzu Circuit, studie optimalizace hloubky a pochopení efektů transpilace před zahájením plného spuštění.
## Začínáme
V této dokumentaci projdeme kroky pro použití nástroje Iskay Quantum Optimizer. Ukážeme si, jak načíst funkci z katalogu a jak převést svůj problém na platný vstup, přičemž si zároveň ukážeme, jak můžeš experimentovat s různými volitelnými parametry.

Podrobnější příklad najdeš v tutoriálu [Řešení problému Market Split s nástrojem Iskay Quantum Optimizer od Kipu Quantum](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer), kde procházíme celý proces použití Iskay Solveru k řešení problému Market Split – reálné výzvy v oblasti alokace zdrojů, kde je třeba rozdělit trhy do vyvážených prodejních regionů tak, aby byly splněny přesné cíle poptávky.

Ověř svoji totožnost pomocí API klíče, který najdeš na [řídicím panelu IBM Quantum Platform](http://quantum.cloud.ibm.com/), a vyber Qiskit Function následovně:

In [ ]:
# ruff: noqa: F821

> **Note:** Následující kód předpokládá, že máš uložené přihlašovací údaje. Pokud ne, postupuj podle pokynů v části [uložení účtu IBM Cloud](/guides/functions#install-qiskit-functions-catalog-client) a ověř se svým API klíčem.

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(
    channel="ibm_quantum_platform",
    instance="INSTANCE_CRN",
    token="YOUR_API_KEY",  # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
)

# Access Function
optimizer = catalog.load("kipu-quantum/iskay-quantum-optimizer")

## Příklad 1: Jednoduchá účelová funkce

Uvažujme účelovou funkci ve spinové formulaci:
$$
C(x_0, x_1, x_2, x_3, x_4) = 1 + 1.5x_0 + 2x_1 + 1.3x_2 + 2.5x_0x_3 + 3.5x_1x_4 + 4x_0x_1x_2
$$

kde $(x_0, ..., x_4) \in {-1, 1}^5$.

Řešení této jednoduché účelové funkce je
$$
(x_0, x_1, x_2, x_3, x_4) = (-1, -1, -1, 1, 1)
$$
s minimální hodnotou $C^{*} = -6$

### 1. Vytvoření účelové funkce

Začneme vytvořením slovníku s koeficienty účelové funkce:

In [ ]:
objective_func = {
    "()": 1,
    "(0,)": 1.5,
    "(1,)": 2,
    "(2,)": 1.3,
    "(0, 3)": 2.5,
    "(1, 4)": 3.5,
    "(0, 1, 2)": 4,
}

### 2. Spuštění optimalizátoru
Problém vyřešíme spuštěním optimalizátoru. Protože $(x_0, ..., x_4) \in {-1, 1}^5$, musíme nastavit `problem_type=spin`.

In [ ]:
# Setup options to run the optimizer
options = {"shots": 5000, "num_iterations": 5, "use_session": True}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": backend_name,  # such as "ibm_fez"
    "options": options,
}

job = optimizer.run(**arguments)

### 3. Získání výsledku
Řešení optimalizačního problému je poskytnuto přímo z optimalizátoru.

In [ ]:
print(job.result())

Zobrazí se slovník ve tvaru:

```
{'solution': {'0': -1, '1': -1, '2': -1, '3': 1, '4': 1},
 'solution_info': {'bitstring': '11100',
  'cost': -13.8,
  'seed_transpiler': 42,
  'mapping': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}},
 'prob_type': 'spin'}
```

Všimni si, že slovník `solution` zobrazuje vektor výsledku $(x_0, x_1, x_2, x_3, x_4) = (-1, -1, -1, 1, 1)$.

## Příklad 2: MaxCut

Mnohé grafové problémy, jako je MaxCut nebo Maximum independent set, jsou NP-těžké problémy a ideální kandidáti pro testování kvantových algoritmů a hardwaru. Tento příklad demonstruje řešení problému MaxCut na 3-regulárním grafu pomocí Quantum Optimizer.

Aby bylo možné tento příklad spustit, musíš kromě `qiskit-ibm-catalog` nainstalovat také balíček `networkx`. Pro instalaci spusť následující příkaz:

In [ ]:
# %pip install networkx numpy

### 1. Vytvoření účelové funkce
Začni vygenerováním náhodného 3-regulárního grafu. Pro tento graf definujeme účelovou funkci problému MaxCut.

In [ ]:
import networkx as nx

# Create a random 3-regular graph
G = nx.random_regular_graph(3, 10, seed=42)


# Create the objective function for MaxCut in Ising formulation
def graph_to_ising_maxcut(G):
    """
    Convert a NetworkX graph to an Ising Hamiltonian for the Max-Cut problem.
    Args:
        G (networkx.Graph): The input graph.
    Returns:
        dict: The objective function of the Ising model
    """
    # Initialize the linear and quadratic coefficients
    objective_func = {}
    # Populate the coefficients
    for i, j in G.edges:
        objective_func[f"({i}, {j})"] = 0.5
    return objective_func


objective_func = graph_to_ising_maxcut(G)

### 2. Spuštění Optimizeru
Vyřeš problém spuštěním optimizeru.

In [ ]:
options = {"shots": 5000, "num_iterations": 5, "use_session": True}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": backend_name,  # such as "ibm_fez"
    "options": options,
}

job = optimizer.run(**arguments)

### 3. Získání výsledku
Získej výsledek a namapuj bitový řetězec řešení zpět na původní uzly grafu.

In [ ]:
print(job.result())

Řešení problému MaxCut je přímo obsaženo v sub-slovníku `solution` výsledného objektu.

In [ ]:
maxcut_solution = job.result()["solution"]

## Příklad 3: Benchmarkové instance
Benchmarkové instance jsou dostupné na GitHubu: [Kipu benchmark instances.](https://github.com/Kipu-Quantum-GmbH/benchmark-instances)

Instance lze načíst pomocí knihovny `pygithub`. Pro instalaci spusť následující příkaz:

In [ ]:
# %pip install pygithub

Cesty k benchmarkovým instancím jsou:

**Maxcut:**
- `'maxcut/maxcut_28_nodes.json'`
- `'maxcut/maxcut_30_nodes.json'`
- `'maxcut/maxcut_32_nodes.json'`
- `'maxcut/maxcut_80_nodes.json'`
- `'maxcut/maxcut_100_nodes.json'`
- `'maxcut/maxcut_120_nodes.json'`

**HUBO:**
- `'HUBO/hubo1_marrakesh.json'`
- `'HUBO/hubo2_marrakesh.json'`

Pro reprodukci výkonu benchmarku pro HUBO instance zvol Backend `ibm_marrakesh` a nastav `direct_qubit_mapping` na `True` v sub-slovníku `options`.
Následující příklad spustí instanci Maxcut s 32 uzly.

In [ ]:
from github import Github
import urllib
import json
import ast

repo = "Kipu-Quantum-GmbH/benchmark-instances"
path = "maxcut/maxcut_32_nodes.json"
gh = Github()
repo = gh.get_repo(repo)
branch = "main"
file = repo.get_contents(urllib.parse.quote(path), ref=branch)

# load json file with benchmark problem
problem_json = json.loads(file.decoded_content)

# convert objective function to compatible format
objective_func = {
    key: ast.literal_eval(value) for key, value in problem_json.items()
}


# Setup configuration to run the optimizer
options = {
    "shots": 5_000,
    "num_iterations": 5,
    "use_session": True,
    "direct_qubit_mapping": False,
}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": "ibm_brisbane",
    "options": options,
}

job = optimizer.run(**arguments)

result = job.result()

## Případy použití
Typickými případy použití pro Optimization solver jsou problémy kombinatorické optimalizace. Problémy lze řešit z mnoha odvětví, jako jsou finance, farmaceutický průmysl nebo logistika. Několik příkladů:
* Optimalizace portfolia (QUBO): [vědecká publikace](https://doi.org/10.1103/PhysRevApplied.22.054037) a [white paper](https://kipu-quantum.com/zope64/kipu_2024/content/e3915/e3916/e4187/White-Paper-2-Financial-modeling-on-quantum-computers-using-digitally-compressed-algorithms-1.pdf)
* Skládání proteinů (HUBO): [vědecká publikace](https://doi.org/10.1103/PhysRevApplied.20.014024)
* Plánování v logistice (QUBO): [vědecká publikace](https://doi.org/10.1103/PhysRevApplied.22.064068)
* Optimalizace sítí: [webinář](https://www.youtube.com/watch?v=w5SrCIK88No)
* Rozdělení trhu (QUBO): [tutoriál](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer)

Pokud máš zájem řešit konkrétní případ použití a vyvinout pro něj dedikované mapování, můžeme ti pomoci. [Kontaktuj nás.](https://share-eu1.hsforms.com/2Ff8cgWvTR9ukT_fPoaNhDw2dqpz5)
## Získání podpory
Pro podporu kontaktuj support@kipu-quantum.com.
## Další kroky
- [Požádej o přístup k Quantum Optimizer od Kipu Quantum](https://share-eu1.hsforms.com/2Ff8cgWvTR9ukT_fPoaNhDw2dqpz5)
- Vyzkoušej tutoriál [Řešení problému rozdělení trhu s Iskay Quantum Optimizer od Kipu Quantum](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer).
- Prostuduj [Romero, S. V., et al. (2025).  Bias-Field Digitized Counterdiabatic Quantum Algorithm for Higher-Order Binary Optimization. arXiv preprint arXiv:2409.04477.](https://arxiv.org/abs/2409.04477)
- Prostuduj [Cadavid, A. G., et al. (2024).  Bias-field digitized counterdiabatic quantum optimization. arXiv preprint arXiv:2405.13898.](https://arxiv.org/abs/2405.13898)
- Prostuduj [Chandarana, P., et al. (2025).  Runtime Quantum Advantage with Digital Quantum Optimization. arXiv preprint arXiv:2505.08663.](https://arxiv.org/abs/2505.08663)
## Další informace